# CNN mit Keras/TensorFlow zur Erkennung von Autos

Dieses Notebook implementiert ein Convolutional Neural Network (CNN) mit Keras/TensorFlow zur Erkennung von Autos im CIFAR-10 Datensatz.

## Einführung und theoretischer Hintergrund

Convolutional Neural Networks (CNNs) haben sich als äußerst effektiv für Bilderkennungsaufgaben erwiesen. Im Gegensatz zu herkömmlichen neuronalen Netzwerken nutzen CNNs spezielle Schichten, die lokale Muster in Bildern erkennen können, ähnlich wie das menschliche visuelle System.

Die Hauptkomponenten eines CNN sind:

1. **Convolutional Layer (Faltungsschicht)**: Diese Schicht wendet Filteroperationen auf das Eingabebild an, um Merkmale wie Kanten, Texturen und Formen zu extrahieren. Jeder Filter erzeugt eine Feature Map, die bestimmte Merkmale im Bild hervorhebt.

2. **Pooling Layer (Pooling-Schicht)**: Diese Schicht reduziert die räumliche Dimension der Feature Maps, was die Berechnungseffizienz erhöht und eine gewisse Translationsinvarianz einführt. Max-Pooling ist eine häufig verwendete Methode, bei der der maximale Wert aus einem definierten Bereich ausgewählt wird.

3. **Fully Connected Layer (Vollständig verbundene Schicht)**: Diese Schicht verbindet alle Neuronen mit allen Neuronen der vorherigen Schicht und wird typischerweise am Ende des Netzwerks verwendet, um die extrahierten Merkmale für die Klassifikation zu nutzen.

4. **Aktivierungsfunktionen**: Funktionen wie ReLU (Rectified Linear Unit) führen Nichtlinearität in das Netzwerk ein, was es ermöglicht, komplexe Muster zu lernen.

5. **Dropout**: Eine Regularisierungstechnik, die während des Trainings zufällig Neuronen deaktiviert, um Overfitting zu reduzieren.

In diesem Notebook verwenden wir Keras, eine benutzerfreundliche API für TensorFlow, um ein CNN zu implementieren, das Autos in Bildern des CIFAR-10 Datensatzes erkennen kann. Keras bietet eine intuitive Schnittstelle zum Erstellen und Trainieren von neuronalen Netzwerken, was die Entwicklung und das Experimentieren mit verschiedenen Architekturen erleichtert.

## Überblick über die Schritte
- Laden der vorbereiteten Daten aus dem vorherigen Notebook
- Definition eines CNN-Modells mit Keras
- Training des Modells mit Early Stopping und Checkpointing
- Evaluierung des Modells auf Testdaten
- Visualisierung des Trainingsverlaufs und der Ergebnisse

## Importieren der benötigten Bibliotheken

Für die Implementierung unseres CNN benötigen wir verschiedene Python-Bibliotheken:

- **numpy**: Für effiziente numerische Operationen und Array-Manipulationen
- **matplotlib**: Für die Visualisierung der Trainingsergebnisse und Vorhersagen
- **tensorflow** und **keras**: Für die Definition, das Training und die Evaluierung des CNN-Modells
- **sklearn**: Für Evaluierungsmetriken und die Konfusionsmatrix
- **os**: Für Dateisystem-Operationen wie das Erstellen von Verzeichnissen

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
import seaborn as sns
import os

## Vorbereitung der Verzeichnisse

Bevor wir mit der Modellierung beginnen, erstellen wir Verzeichnisse für die Speicherung der Modelle und Visualisierungen. Eine gute Organisation der Projektstruktur ist wichtig für die Nachvollziehbarkeit und Wiederverwendbarkeit des Codes.

Wir erstellen zwei Verzeichnisse:
- `models_dir`: Für die Speicherung der trainierten Modelle und Checkpoints
- `visualizations_dir`: Für die Speicherung von Visualisierungen wie Lernkurven und Konfusionsmatrizen

Die Funktion `os.makedirs()` mit dem Parameter `exist_ok=True` stellt sicher, dass kein Fehler auftritt, falls die Verzeichnisse bereits existieren.

In [2]:
# Vorbereitung der Verzeichnisse für Modelle und Visualisierungen
data_dir = '../data'
models_dir = '../models'
visualizations_dir = '../models/visualizations'

os.makedirs(models_dir, exist_ok=True)
os.makedirs(visualizations_dir, exist_ok=True)

## Laden der vorbereiteten Daten

Im vorherigen Notebook haben wir den CIFAR-10 Datensatz vorbereitet und die normalisierten Bilddaten sowie die binären Labels gespeichert. Jetzt laden wir diese vorbereiteten Daten, um unser CNN-Modell zu trainieren.

Die Daten bestehen aus:
- `x_train_normalized`: Normalisierte Trainingsbilder (Werte im Bereich [0, 1])
- `x_test_normalized`: Normalisierte Testbilder
- `y_train_binary`: Binäre Labels für die Trainingsbilder (1 für Auto, 0 für Nicht-Auto)
- `y_test_binary`: Binäre Labels für die Testbilder

Das Laden der vorbereiteten Daten spart Zeit und Rechenressourcen, da wir die Vorverarbeitung nicht erneut durchführen müssen.

In [3]:
# Laden der vorbereiteten Daten
x_train = np.load(os.path.join(data_dir, 'x_train_normalized.npy'))
x_test = np.load(os.path.join(data_dir, 'x_test_normalized.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train_binary.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test_binary.npy'))

# Überprüfen der geladenen Daten
print(f"Trainingsbilder: {x_train.shape}, Wertebereich: [{np.min(x_train)}, {np.max(x_train)}]")
print(f"Testbilder: {x_test.shape}, Wertebereich: [{np.min(x_test)}, {np.max(x_test)}]")
print(f"Trainings-Labels: {y_train.shape}, Klassen: {np.unique(y_train)}")
print(f"Test-Labels: {y_test.shape}, Klassen: {np.unique(y_test)}")

# Klassenverteilung anzeigen
train_class_counts = {int(label): np.sum(y_train == label) for label in np.unique(y_train)}
test_class_counts = {int(label): np.sum(y_test == label) for label in np.unique(y_test)}
print(f"Klassenverteilung im Trainingsdatensatz: {train_class_counts}")
print(f"Klassenverteilung im Testdatensatz: {test_class_counts}")

Trainingsbilder: (50000, 32, 32, 3), Wertebereich: [0.0, 1.0]
Testbilder: (10000, 32, 32, 3), Wertebereich: [0.0, 1.0]
Trainings-Labels: (50000, 1), Klassen: [0 1]
Test-Labels: (10000, 1), Klassen: [0 1]
Klassenverteilung im Trainingsdatensatz: {0: 45000, 1: 5000}
Klassenverteilung im Testdatensatz: {0: 9000, 1: 1000}


## Definition des CNN-Modells

Jetzt definieren wir die Architektur unseres CNN-Modells. Wir verwenden ein sequentielles Modell in Keras, das eine lineare Abfolge von Schichten darstellt.

Unsere Architektur besteht aus:

1. **Convolutional Layers**: Wir verwenden mehrere Faltungsschichten mit zunehmender Anzahl von Filtern (32, 64, 128), um hierarchische Merkmale zu extrahieren. Jede Faltungsschicht verwendet einen 3x3 Kernel und die ReLU-Aktivierungsfunktion.

2. **Max Pooling Layers**: Nach jeder Faltungsschicht verwenden wir eine Max-Pooling-Schicht mit einem 2x2 Pool-Fenster, um die räumliche Dimension zu reduzieren und die wichtigsten Merkmale beizubehalten.

3. **Flatten Layer**: Diese Schicht wandelt die mehrdimensionalen Feature Maps in einen eindimensionalen Vektor um, der als Eingabe für die vollständig verbundenen Schichten dient.

4. **Dense Layers (Fully Connected)**: Wir verwenden zwei vollständig verbundene Schichten. Die erste hat 128 Neuronen mit ReLU-Aktivierung, und die zweite (Ausgabeschicht) hat ein Neuron mit Sigmoid-Aktivierung für die binäre Klassifikation.

5. **Dropout**: Zwischen den vollständig verbundenen Schichten fügen wir eine Dropout-Schicht mit einer Rate von 0.5 ein, um Overfitting zu reduzieren.

Diese Architektur ist für die Erkennung von Autos in den CIFAR-10 Bildern geeignet, da sie sowohl einfache als auch komplexe Merkmale erfassen kann und gleichzeitig Overfitting durch Regularisierung verhindert.

In [4]:
# Definition des CNN-Modells
def create_cnn_model(input_shape=(32, 32, 3)):
    model = Sequential([
        # Erste Convolutional Layer
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        MaxPooling2D((2, 2)),
        
        # Zweite Convolutional Layer
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        
        # Dritte Convolutional Layer
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        
        # Flatten Layer
        Flatten(),
        
        # Fully Connected Layers
        Dense(128, activation='relu'),
        Dropout(0.5),  # Dropout zur Reduzierung von Overfitting
        Dense(1, activation='sigmoid')  # Ausgabeschicht für binäre Klassifikation
    ])
    
    return model

## Modell erstellen und Zusammenfassung anzeigen

Wir erstellen nun eine Instanz unseres CNN-Modells und kompilieren es mit geeigneten Hyperparametern:

- **Optimizer**: Wir verwenden den Adam-Optimizer, der adaptive Lernraten für jeden Parameter bietet und in der Praxis gut funktioniert.
- **Loss Function**: Für die binäre Klassifikation verwenden wir die Binary Cross-Entropy Loss-Funktion.
- **Metrics**: Wir verfolgen die Accuracy (Genauigkeit) während des Trainings, um die Leistung des Modells zu überwachen.

Nach der Kompilierung zeigen wir eine Zusammenfassung des Modells an, die die Architektur, die Anzahl der Parameter und die Form der Ausgabe jeder Schicht darstellt. Dies gibt uns einen guten Überblick über die Komplexität des Modells und hilft, potenzielle Probleme zu identifizieren.

In [5]:
# Modell erstellen
model = create_cnn_model()

# Modell kompilieren
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Modellzusammenfassung anzeigen
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 32)        896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 16, 16, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 16, 16, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 8, 8, 64)         0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           (None, 8, 8, 128)         73856     
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 4, 4, 128)        0

## Callbacks für das Training

Callbacks sind Funktionen, die während des Trainings zu bestimmten Zeitpunkten aufgerufen werden. Sie können verwendet werden, um das Training zu überwachen, zu steuern oder zu protokollieren. Wir verwenden zwei wichtige Callbacks:

1. **Early Stopping**: Dieser Callback überwacht eine bestimmte Metrik (in unserem Fall die Validierungs-Loss) und stoppt das Training, wenn sich diese Metrik über eine bestimmte Anzahl von Epochen nicht verbessert. Dies verhindert Overfitting und spart Rechenzeit.

2. **Model Checkpoint**: Dieser Callback speichert das Modell nach jeder Epoche, wenn sich eine bestimmte Metrik verbessert hat. Wir speichern das Modell, wenn die Validierungs-Loss sinkt, und behalten so das beste Modell während des Trainings.

Diese Callbacks sind besonders nützlich für das Training von neuronalen Netzwerken, da sie helfen, die optimale Anzahl von Trainingsepochen zu finden und das beste Modell zu speichern.

In [6]:
# Callbacks für das Training
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

checkpoint_path = os.path.join(models_dir, 'cnn_keras_best_model.h5')
model_checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

callbacks = [early_stopping, model_checkpoint]

## Training des Modells

Jetzt trainieren wir unser CNN-Modell auf den vorbereiteten Daten. Während des Trainings wird das Modell die Gewichte anpassen, um die Vorhersagefehler zu minimieren.

Wichtige Parameter für das Training sind:

- **Batch Size**: Die Anzahl der Beispiele, die in einem Schritt verarbeitet werden. Ein größerer Batch führt zu einer stabileren Gradientenschätzung, benötigt aber mehr Speicher.
- **Epochs**: Die Anzahl der vollständigen Durchläufe durch den Trainingsdatensatz. Wir setzen eine hohe Zahl, aber Early Stopping wird das Training stoppen, wenn keine Verbesserung mehr auftritt.
- **Validation Split**: Der Anteil der Trainingsdaten, der für die Validierung während des Trainings verwendet wird. Dies hilft, Overfitting zu erkennen.
- **Callbacks**: Die zuvor definierten Callbacks für Early Stopping und Model Checkpointing.

Das Training kann je nach Hardware einige Zeit in Anspruch nehmen. Die Fortschrittsanzeige zeigt den Verlust und die Genauigkeit für jede Epoche sowohl für die Trainings- als auch für die Validierungsdaten.

In [7]:
# Training des Modells
batch_size = 32
epochs = 50
validation_split = 0.2

history = model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=validation_split,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
1250/1250 [==============================] - ETA: 0s - loss: 0.3029 - accuracy: 0.8748
Epoch 1: val_loss improved from inf to 0.21903, saving model to ../models/cnn_keras_best_model.h5
1250/1250 [==============================] - 14s 11ms/step - loss: 0.3029 - accuracy: 0.8748 - val_loss: 0.2190 - val_accuracy: 0.9132
Epoch 2/50
1250/1250 [==============================] - ETA: 0s - loss: 0.2001 - accuracy: 0.9196
Epoch 2: val_loss improved from 0.21903 to 0.18347, saving model to ../models/cnn_keras_best_model.h5
1250/1250 [==============================] - 14s 11ms/step - loss: 0.2001 - accuracy: 0.9196 - val_loss: 0.1835 - val_accuracy: 0.9300
Epoch 3/50
1250/1250 [==============================] - ETA: 0s - loss: 0.1676 - accuracy: 0.9346
Epoch 3: val_loss improved from 0.18347 to 0.16926, saving model to ../models/cnn_keras_best_model.h5
1250/1250 [==============================] - 14s 11ms/step - loss: 0.1676 - accuracy: 0.9346 - val_loss: 0.1693 - val_accuracy: 0.9352

## Speichern des Modells

Obwohl wir bereits das beste Modell während des Trainings mit dem ModelCheckpoint-Callback gespeichert haben, speichern wir hier das endgültige Modell explizit. Dies ist nützlich, wenn wir später das Modell für Vorhersagen verwenden möchten, ohne es erneut trainieren zu müssen.

Das Modell wird im HDF5-Format (.h5) gespeichert, das sowohl die Architektur als auch die Gewichte des Modells enthält. Dies ermöglicht ein einfaches Laden und Verwenden des Modells in anderen Anwendungen.

In [8]:
# Speichern des finalen Modells
final_model_path = os.path.join(models_dir, 'cnn_keras_final_model.h5')
model.save(final_model_path)
print(f"Modell wurde gespeichert unter: {final_model_path}")

Modell wurde gespeichert unter: ../models/cnn_keras_final_model.h5


## Evaluierung des Modells auf den Testdaten

Nach dem Training evaluieren wir das Modell auf den Testdaten, um seine Generalisierungsfähigkeit zu bewerten. Die Testdaten wurden während des Trainings nicht verwendet, daher geben sie uns eine unvoreingenommene Einschätzung der Modellleistung.

Wir berechnen den Verlust und die Genauigkeit auf den Testdaten und machen Vorhersagen, die wir später für detailliertere Analysen verwenden werden. Die Vorhersagen sind Wahrscheinlichkeitswerte zwischen 0 und 1, die wir mit einem Schwellenwert (typischerweise 0.5) in binäre Klassen umwandeln können.

In [9]:
# Evaluierung des Modells auf den Testdaten
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print("Evaluierung auf Testdaten:")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f}\n")

# Vorhersagen für die Testdaten
y_pred_prob = model.predict(x_test)
y_pred = (y_pred_prob > 0.5).astype(int)
print("Vorhersagen wurden erstellt.")

313/313 [==============================] - 1s 3ms/step - loss: 0.1196 - accuracy: 0.9636
Evaluierung auf Testdaten:
Loss: 0.1196
Accuracy: 0.9636

313/313 [==============================] - 1s 3ms/step
Vorhersagen wurden erstellt.


## Visualisierung des Trainingsverlaufs

Die Visualisierung des Trainingsverlaufs hilft uns, das Verhalten des Modells während des Trainings zu verstehen. Wir plotten den Verlust und die Genauigkeit sowohl für die Trainings- als auch für die Validierungsdaten über die Epochen hinweg.

Diese Plots können uns wichtige Einblicke geben:
- Konvergiert das Modell? (Sinkt der Verlust kontinuierlich?)
- Gibt es Anzeichen von Overfitting? (Divergieren die Trainings- und Validierungskurven?)
- Wie viele Epochen sind optimal? (Wann beginnt der Validierungsverlust zu steigen?)

Die Visualisierungen werden auch gespeichert, um sie später in Berichten oder Präsentationen verwenden zu können.

In [10]:
# Visualisierung des Trainingsverlaufs
plt.figure(figsize=(12, 5))

# Plot für den Verlust
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Trainingsverlauf - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot für die Genauigkeit
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Trainingsverlauf - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(visualizations_dir, 'cnn_keras_training_history.png'))
plt.close()

print("Trainingsverlauf wurde visualisiert und gespeichert.")

Trainingsverlauf wurde visualisiert und gespeichert.


## Detaillierte Evaluierung mit Precision, Recall und F1-Score

Für eine umfassendere Bewertung des Modells berechnen wir zusätzliche Metriken wie Precision, Recall und F1-Score. Diese Metriken sind besonders wichtig bei unausgewogenen Datensätzen, wie es bei unserem binären Klassifikationsproblem der Fall ist (10% Autos, 90% Nicht-Autos).

- **Precision (Präzision)**: Der Anteil der korrekt als positiv klassifizierten Beispiele an allen als positiv klassifizierten Beispielen. Precision = TP / (TP + FP)
- **Recall (Sensitivität)**: Der Anteil der korrekt als positiv klassifizierten Beispiele an allen tatsächlich positiven Beispielen. Recall = TP / (TP + FN)
- **F1-Score**: Das harmonische Mittel aus Precision und Recall. F1 = 2 * (Precision * Recall) / (Precision + Recall)

Wir verwenden die Funktionen aus scikit-learn, um diese Metriken zu berechnen und einen detaillierten Klassifikationsbericht zu erstellen.

In [11]:
# Detaillierte Evaluierung mit Precision, Recall und F1-Score
print("Detaillierte Evaluierungsmetriken:\n")
print(classification_report(y_test, y_pred))

# Berechnung von Precision, Recall und F1-Score für die positive Klasse (Autos)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average=None)
print(f"Precision: {precision[1]:.4f}")
print(f"Recall: {recall[1]:.4f}")
print(f"F1-Score: {f1[1]:.4f}")

Detaillierte Evaluierungsmetriken:

              precision    recall  f1-score   support

           0       0.97      0.99      0.98      9000
           1       0.91      0.76      0.83      1000

    accuracy                           0.96     10000
   macro avg       0.94      0.88      0.90     10000
weighted avg       0.96      0.96      0.96     10000

Precision: 0.9143
Recall: 0.7630
F1-Score: 0.8318


## Visualisierung der Konfusionsmatrix

Die Konfusionsmatrix ist eine nützliche Visualisierung, die zeigt, wie viele Beispiele jeder Klasse korrekt und falsch klassifiziert wurden. Sie gibt uns einen detaillierten Einblick in die Leistung des Modells für jede Klasse.

Die Konfusionsmatrix enthält vier Werte:
- **True Positives (TP)**: Autos, die korrekt als Autos klassifiziert wurden
- **False Positives (FP)**: Nicht-Autos, die fälschlicherweise als Autos klassifiziert wurden
- **True Negatives (TN)**: Nicht-Autos, die korrekt als Nicht-Autos klassifiziert wurden
- **False Negatives (FN)**: Autos, die fälschlicherweise als Nicht-Autos klassifiziert wurden

Diese Visualisierung hilft uns, die Stärken und Schwächen des Modells besser zu verstehen und mögliche Verbesserungen zu identifizieren.

In [12]:
# Berechnung der Konfusionsmatrix
cm = confusion_matrix(y_test, y_pred)
print("Konfusionsmatrix:")
print(cm)
print()

# Visualisierung der Konfusionsmatrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Nicht-Auto', 'Auto'],
            yticklabels=['Nicht-Auto', 'Auto'])
plt.title('Konfusionsmatrix')
plt.ylabel('Tatsächliche Klasse')
plt.xlabel('Vorhergesagte Klasse')
plt.tight_layout()
plt.savefig(os.path.join(visualizations_dir, 'cnn_keras_confusion_matrix.png'))
plt.close()

print("Konfusionsmatrix wurde visualisiert und gespeichert.")

Konfusionsmatrix:
[[8933   67]
 [ 237  763]]

Konfusionsmatrix wurde visualisiert und gespeichert.


## Visualisierung einiger Vorhersagen

Um ein besseres Verständnis für die Leistung unseres Modells zu bekommen, visualisieren wir einige Beispiele aus dem Testdatensatz zusammen mit den Vorhersagen des Modells. Wir zeigen sowohl korrekte als auch falsche Vorhersagen, um zu verstehen, wo das Modell gut funktioniert und wo es Schwierigkeiten hat.

Diese Visualisierung kann uns helfen, mögliche Muster in den Fehlern des Modells zu erkennen und Ideen für Verbesserungen zu entwickeln. Zum Beispiel könnten wir feststellen, dass das Modell Schwierigkeiten hat, Autos aus bestimmten Perspektiven oder unter bestimmten Lichtbedingungen zu erkennen.

In [13]:
# Visualisierung einiger Vorhersagen
def visualize_predictions(x_data, y_true, y_pred, y_pred_prob, num_examples=10, save_path=None):
    # Zufällige Indizes auswählen
    np.random.seed(42)  # Für Reproduzierbarkeit
    indices = np.random.choice(len(y_true), size=num_examples, replace=False)
    
    # Erstellen der Visualisierung
    plt.figure(figsize=(15, 8))
    for i, idx in enumerate(indices):
        plt.subplot(2, 5, i+1)
        plt.imshow(x_data[idx])
        plt.axis('off')
        
        true_label = 'Auto' if y_true[idx][0] == 1 else 'Nicht-Auto'
        pred_label = 'Auto' if y_pred[idx][0] == 1 else 'Nicht-Auto'
        confidence = y_pred_prob[idx][0]
        
        color = 'green' if y_true[idx][0] == y_pred[idx][0] else 'red'
        plt.title(f"Wahr: {true_label}\nVorhersage: {pred_label}\nKonfidenz: {confidence:.2f}", color=color)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

# Visualisierung von korrekten Vorhersagen
correct_indices = np.where((y_test == y_pred) & (y_test == 1))[0]  # Korrekt klassifizierte Autos
visualize_predictions(
    x_test, y_test, y_pred, y_pred_prob, 
    num_examples=5, 
    save_path=os.path.join(visualizations_dir, 'cnn_keras_correct_predictions.png')
)

# Visualisierung von falschen Vorhersagen
incorrect_indices = np.where(y_test != y_pred)[0]  # Falsch klassifizierte Beispiele
if len(incorrect_indices) > 0:
    visualize_predictions(
        x_test[incorrect_indices], y_test[incorrect_indices], y_pred[incorrect_indices], y_pred_prob[incorrect_indices], 
        num_examples=min(5, len(incorrect_indices)), 
        save_path=os.path.join(visualizations_dir, 'cnn_keras_incorrect_predictions.png')
    )

print("Vorhersagen wurden visualisiert und gespeichert.")

Vorhersagen wurden visualisiert und gespeichert.


## Zusammenfassung

In diesem Notebook haben wir ein Convolutional Neural Network (CNN) mit Keras/TensorFlow implementiert, um Autos im CIFAR-10 Datensatz zu erkennen. Hier sind die wichtigsten Schritte und Ergebnisse:

1. **Datenaufbereitung**: Wir haben die vorbereiteten Daten aus dem vorherigen Notebook geladen, die bereits normalisiert und mit binären Labels versehen waren.

2. **Modellarchitektur**: Wir haben ein CNN mit mehreren Faltungsschichten, Max-Pooling, Dropout und vollständig verbundenen Schichten definiert. Die Architektur wurde speziell für die Erkennung von Autos in den CIFAR-10 Bildern entworfen.

3. **Training**: Wir haben das Modell mit dem Adam-Optimizer und der Binary Cross-Entropy Loss-Funktion trainiert. Wir haben Early Stopping und Model Checkpointing verwendet, um Overfitting zu vermeiden und das beste Modell zu speichern.

4. **Evaluierung**: Wir haben das Modell auf den Testdaten evaluiert und eine Genauigkeit von etwa 96% erreicht. Wir haben auch detailliertere Metriken wie Precision, Recall und F1-Score berechnet, die ein umfassenderes Bild der Modellleistung geben.

5. **Visualisierung**: Wir haben den Trainingsverlauf, die Konfusionsmatrix und einige Beispielvorhersagen visualisiert, um ein besseres Verständnis für die Leistung des Modells zu bekommen.

Das trainierte Modell zeigt eine gute Leistung bei der Erkennung von Autos, mit einer hohen Genauigkeit und einem guten F1-Score. Die Visualisierungen zeigen, dass das Modell in den meisten Fällen korrekte Vorhersagen trifft, aber auch einige Fehler macht, insbesondere bei schwierigeren Beispielen.

Im nächsten Notebook werden wir ein benutzerdefiniertes CNN ohne die Verwendung von keras.models oder keras.layers implementieren, um ein tieferes Verständnis für die zugrunde liegenden Operationen zu bekommen.